In [ ]:
import numpy as np
import pandas as pd
import scipy.stats as stats
import matplotlib.pyplot as plt
import datetime

np.random.seed(42)


num_days = 1260
date_range = pd.date_range(end=datetime.date.today(), periods=num_days, freq='B')


nifty50_pct = np.random.normal(0.0005, 0.01, num_days) # ~12% annualized
nifty100_pct = nifty50_pct + np.random.normal(0.00005, 0.002, num_days) # Highly correlated

nifty50_nav = 10000 * np.cumprod(1 + nifty50_pct)
nifty100_nav = 10000 * np.cumprod(1 + nifty100_pct)

nav_df = pd.DataFrame(index=date_range)
nav_df['Nifty_50'] = nifty50_nav
nav_df['Nifty_100'] = nifty100_nav


fund_names = [f'Fund_Scheme_{i+1}' for i in range(40)]
expense_ratios = np.random.uniform(0.005, 0.025, 40) 
for fund in fund_names:
    
    beta = np.random.uniform(0.7, 1.3)
    alpha = np.random.uniform(-0.0002, 0.0005)
    noise = np.random.normal(0, 0.005, num_days)
    fund_returns = (beta * nifty100_pct) + alpha + noise - (0.015 / 252)
    nav_df[fund] = 10 * np.cumprod(1 + fund_returns)


returns_df = nav_df.pct_change().dropna()


print("--- Distribution Validation (Daily Returns) ---")
print(returns_df[fund_names].describe().loc[['mean', 'std', 'min', 'max']])
print("-" * 50)


Rf_annual = 0.065
Rf_daily = Rf_annual / 252

results = []

for fund in fund_names:
    fund_nav = nav_df[fund]
    fund_ret = returns_df[fund]
    
    cagr_1yr = (fund_nav.iloc[-1] / fund_nav.iloc[-252]) ** (252 / 252) - 1 if len(fund_nav) >= 252 else np.nan
    cagr_3yr = (fund_nav.iloc[-1] / fund_nav.iloc[-756]) ** (252 / 756) - 1 if len(fund_nav) >= 756 else np.nan
    cagr_5yr = (fund_nav.iloc[-1] / fund_nav.iloc[0]) ** (252 / len(fund_nav)) - 1
    
    
    excess_ret = fund_ret - Rf_daily
    sharpe = (excess_ret.mean() / fund_ret.std()) * np.sqrt(252) if fund_ret.std() != 0 else 0
    
    downside_ret = fund_ret[fund_ret < 0]
    sortino = (excess_ret.mean() / downside_ret.std()) * np.sqrt(252) if len(downside_ret) > 0 else 0
    
    beta, alpha_daily, r_value, p_value, std_err = stats.linregress(returns_df['Nifty_100'], fund_ret)
    alpha_annual = alpha_daily * 252
    
    running_max = fund_nav.cummax()
    drawdown = (fund_nav / running_max) - 1
    max_dd = drawdown.min()
    
    worst_dd_date = drawdown.idxmin()
    
    peak_date = fund_nav.loc[:worst_dd_date].idxmax()
    
   
    tracking_error = (fund_ret - returns_df['Nifty_50']).std() * np.sqrt(252)
    
    results.append({
        'Fund': fund,
        'CAGR_1Yr': cagr_1yr,
        'CAGR_3Yr': cagr_3yr,
        'CAGR_5Yr': cagr_5yr,
        'Sharpe_Ratio': sharpe,
        'Sortino_Ratio': sortino,
        'Beta': beta,
        'Alpha_Annualized': alpha_annual,
        'Max_Drawdown': max_dd,
        'Max_DD_Start': peak_date.strftime('%Y-%m-%d'),
        'Max_DD_End': worst_dd_date.strftime('%Y-%m-%d'),
        'Tracking_Error_N50': tracking_error,
        'Expense_Ratio': expense_ratios[fund_names.index(fund)]
    })

metrics_df = pd.DataFrame(results)

metrics_df['Rank_3Yr'] = metrics_df['CAGR_3Yr'].rank(pct=True)
metrics_df['Rank_Sharpe'] = metrics_df['Sharpe_Ratio'].rank(pct=True)
metrics_df['Rank_Alpha'] = metrics_df['Alpha_Annualized'].rank(pct=True)

metrics_df['Rank_Expense'] = metrics_df['Expense_Ratio'].rank(pct=True, ascending=False)
metrics_df['Rank_MaxDD'] = metrics_df['Max_Drawdown'].rank(pct=True) 

metrics_df['Fund_Score'] = (
    0.30 * metrics_df['Rank_3Yr'] +
    0.25 * metrics_df['Rank_Sharpe'] +
    0.20 * metrics_df['Rank_Alpha'] +
    0.15 * metrics_df['Rank_Expense'] +
    0.10 * metrics_df['Rank_MaxDD']
) * 100

metrics_df = metrics_df.sort_values(by='Fund_Score', ascending=False).reset_index(drop=True)

alpha_beta_df = metrics_df[['Fund', 'Alpha_Annualized', 'Beta']].copy()
alpha_beta_df.to_csv('alpha_beta.csv', index=False)

scorecard_out = metrics_df[[
    'Fund', 'Fund_Score', 'CAGR_1Yr', 'CAGR_3Yr', 'CAGR_5Yr', 
    'Sharpe_Ratio', 'Sortino_Ratio', 'Max_Drawdown', 'Max_DD_Start', 
    'Max_DD_End', 'Tracking_Error_N50', 'Expense_Ratio'
]].copy()
scorecard_out.to_csv('fund_scorecard.csv', index=False)

print("Saved deliverables: alpha_beta.csv, fund_scorecard.csv")

top_5_funds = metrics_df['Fund'].head(5).tolist()


plot_df = nav_df[['Nifty_50', 'Nifty_100'] + top_5_funds].tail(756).copy()

plot_df = (plot_df / plot_df.iloc[0]) * 100

plt.figure(figsize=(12, 7))
plt.plot(plot_df.index, plot_df['Nifty_50'], label='Nifty 50 (Benchmark)', color='black', linewidth=2, linestyle='--')
plt.plot(plot_df.index, plot_df['Nifty_100'], label='Nifty 100 (Benchmark)', color='gray', linewidth=2, linestyle=':')

for fund in top_5_funds:
    plt.plot(plot_df.index, plot_df[fund], label=f"{fund} (Score: {metrics_df.loc[metrics_df['Fund']==fund, 'Fund_Score'].values[0]:.1f})")

plt.title('Top 5 Performing Funds vs Benchmarks (3-Year Normalized Performance Base 100)', fontsize=14, fontweight='bold')
plt.xlabel('Date', fontsize=12)
plt.ylabel('Normalized Value (Base 100)', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend(loc='upper left')
plt.tight_layout()

plt.savefig('benchmark_comparison_chart.png', dpi=300)
plt.close()
print("Saved deliverable: benchmark_comparison_chart.png")

--- Distribution Validation (Daily Returns) ---
      Fund_Scheme_1  Fund_Scheme_2  Fund_Scheme_3  Fund_Scheme_4  \
mean       0.000955       0.000606       0.001363       0.001002   
std        0.010721       0.011481       0.012493       0.010024   
min       -0.033633      -0.035242      -0.040904      -0.033991   
max        0.036244       0.044814       0.047755       0.034907   

      Fund_Scheme_5  Fund_Scheme_6  Fund_Scheme_7  Fund_Scheme_8  \
mean       0.001547       0.001464       0.000870       0.000623   
std        0.011674       0.012051       0.010262       0.009446   
min       -0.040895      -0.036739      -0.040829      -0.027697   
max        0.039288       0.042623       0.034271       0.036811   

      Fund_Scheme_9  Fund_Scheme_10  ...  Fund_Scheme_31  Fund_Scheme_32  \
mean       0.001445        0.001197  ...        0.001743        0.000951   
std        0.010567        0.012777  ...        0.012713        0.010966   
min       -0.029359       -0.042521  ...  